# Review and label SpecMine repositories

Load a random repository from a 5,000-file sample of the full SpecMine dataset and review each spec ID and its raw Markdown. The sampled files are downloaded into `notebooks/data`; existing files are reused. The notebook displays Markdown as escaped text, so remote images and embedded HTML are not rendered. Add reviewed rows to `examples/synthetic_labels.csv` as `spec_id,label`; use `1` if AI was used at any point, `0` if it was not, and leave the label blank if evidence is ambiguous. The checked-in file contains real, manually verified labels.

This notebook reviews one repository per run. Repeat the review with a different repository before training: the training workflow needs labeled specs from at least two repositories to make a repository-disjoint holdout.

In [ ]:
import html
import sys
from pathlib import Path

from IPython.display import HTML, display

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "specmine").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
from specmine.data import download_spec_files, load_data

print("Project source loaded.")

In [ ]:
SEED = 428
SPEC_SAMPLE_SIZE = 100
DATA_DIR = project_root / "notebooks" / "data"
LABELS_CSV = project_root / "examples" / "synthetic_labels.csv"

specs = load_data(
    curated_only=False,
    sample_size=SPEC_SAMPLE_SIZE,
    seed=SEED,
)
if specs is None:
    raise RuntimeError("SpecMine data loading failed; no specs are available to review.")
if specs.empty:
    raise ValueError("The SpecMine dataset contains no specs to review.")
specs = download_spec_files(specs, data_dir=DATA_DIR)
required_columns = {"repo_name", "file_url_sha16", "local_path"}
missing_columns = required_columns - set(specs.columns)
if missing_columns:
    raise KeyError(f"SpecMine data is missing columns: {sorted(missing_columns)}")


In [ ]:
repositories = specs["repo_name"].dropna().drop_duplicates()
if repositories.empty:
    raise ValueError("The SpecMine sample contains no repository names.")
random_repo = repositories.sample(n=1).iloc[0]
repo_specs = specs.loc[specs["repo_name"].eq(random_repo)].sort_values("file_url_sha16")

print(f"Random repository: {random_repo}")
print(f"Specs to review: {len(repo_specs)}")
print("Labels CSV: examples/synthetic_labels.csv")

In [ ]:
for row in repo_specs.itertuples(index=False):
    spec_id = str(row.file_url_sha16)
    spec_path = Path(row.local_path)
    if not spec_path.is_file():
        raise FileNotFoundError("A downloaded spec file is missing from the data directory.")
    spec_markdown = spec_path.read_text(encoding="utf-8")
    print(f"Repository: {row.repo_name}")
    print(f"spec_id: {spec_id}")
    display(HTML("<pre>" + html.escape(spec_markdown) + "</pre>"))
    display(HTML("<hr>"))

print(
    "Add one CSV row per reviewed spec using its exact spec_id and label "
    "(0, 1, or blank). Repeat this notebook for a different repository "
    "until the labels CSV covers at least two repositories, then train."
)